# Artificial Intelligence Lab - LAB 02
## Experiment 2: Utility-Based Vacuum Cleaner Agent (m × n Grid)
### Objective:
Design a Utility-Based Vacuum Cleaner Agent operating on an $m \times n$ grid with performance metric evaluation:
1. **Fully Observable Environment**: The agent can see every cell's status, computes exact BFS distances to remaining dirt, and selects actions maximizing utility.
2. **Partially Observable Environment**: The agent operates with incomplete knowledge, maintaining a belief state.

### Utility Function:
$$U(a) = \text{VacuumReward}(a) - \text{DistanceToNearestDirty}(a) - \text{MovementCost}(a) + \text{DirtDensityNearby}(a)$$

### 1. Environment and State Representation Functions
- `create_environment(rows, cols)`
- `place_dirt(env, dirt_prob)`
- `place_agent(rows, cols)`
- `is_dirty(env, pos)`
- `vacuum(env, pos)`
- `get_possible_moves(pos, rows, cols)`

In [1]:
import random
from collections import deque

def create_environment(rows, cols):
    """Creates a 2D matrix (rows x cols) filled with 0 (Clean)."""
    return [[0 for _ in range(cols)] for _ in range(rows)]

def place_dirt(env, dirt_prob=0.4):
    """Randomly sets cells to 1 (Dirty) based on dirt_prob."""
    rows, cols = len(env), len(env[0])
    dirt_count = 0
    for r in range(rows):
        for c in range(cols):
            if random.random() < dirt_prob:
                env[r][c] = 1
                dirt_count += 1
    if dirt_count == 0:
        env[0][0] = 1
        env[rows - 1][cols - 1] = 1
    return env

def place_agent(rows, cols):
    """Randomly places agent at a starting (row, col) position."""
    return (random.randint(0, rows - 1), random.randint(0, cols - 1))

def is_dirty(env, pos):
    r, c = pos
    return env[r][c] == 1

def vacuum(env, pos):
    r, c = pos
    env[r][c] = 0
    return env

def get_possible_moves(pos, rows, cols):
    r, c = pos
    moves = {}
    if r > 0:
        moves['Up'] = (r - 1, c)
    if r < rows - 1:
        moves['Down'] = (r + 1, c)
    if c > 0:
        moves['Left'] = (r, c - 1)
    if c < cols - 1:
        moves['Right'] = (r, c + 1)
    return moves

### 2. Distance Computation (BFS) and Utility Evaluation
- `find_nearest_dirty_cell`: Computes shortest BFS distance.
- `get_dirt_density_nearby`: Counts dirty cells in local radius.
- `compute_utility`: Applies the exact utility formula.
- `choose_best_action`: Selects candidate action with argmax $U(a)$.

In [2]:
def find_nearest_dirty_cell(env, pos, rows, cols):
    """Uses BFS shortest-path search to locate the nearest dirty cell and distance."""
    if is_dirty(env, pos):
        return pos, 0

    queue = deque([(pos[0], pos[1], 0)])
    visited = {pos}

    while queue:
        r, c, dist = queue.popleft()
        if env[r][c] == 1:
            return (r, c), dist

        for nr, nc in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc, dist + 1))

    return None, 0

def get_dirt_density_nearby(env, pos, rows, cols, radius=1):
    r, c = pos
    density = 0
    for dr in range(-radius, radius + 1):
        for dc in range(-radius, radius + 1):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                if env[nr][nc] == 1:
                    density += 1
    return density

def compute_utility(action, pos, env, rows, cols):
    if action == 'vacuum':
        resulting_pos = pos
        vacuum_reward = 10 if is_dirty(env, pos) else -5
        movement_cost = 0
    else:
        moves = get_possible_moves(pos, rows, cols)
        resulting_pos = moves[action]
        vacuum_reward = 0
        movement_cost = 1

    _, dist_to_dirty = find_nearest_dirty_cell(env, resulting_pos, rows, cols)
    density = get_dirt_density_nearby(env, resulting_pos, rows, cols, radius=1)

    utility = vacuum_reward - dist_to_dirty - movement_cost + density
    return utility

def choose_best_action(pos, env, rows, cols):
    candidate_actions = []
    if is_dirty(env, pos):
        candidate_actions.append('vacuum')
    else:
        moves = get_possible_moves(pos, rows, cols)
        candidate_actions.extend(moves.keys())

    if not candidate_actions:
        return None

    best_action = None
    best_utility = float('-inf')

    for a in candidate_actions:
        u = compute_utility(a, pos, env, rows, cols)
        if u > best_utility:
            best_utility = u
            best_action = a

    return best_action, best_utility

### 3. Metric Tracking and Visualizers

In [3]:
def all_clean(env):
    return all(cell == 0 for row in env for cell in row)

def compute_theoretical_minimum(env, start_pos, rows, cols):
    dirty_cells = [(r, c) for r in range(rows) for c in range(cols) if env[r][c] == 1]
    if not dirty_cells:
        return 0
    max_dist = max(abs(start_pos[0] - r) + abs(start_pos[1] - c) for r, c in dirty_cells)
    return len(dirty_cells) + max_dist

def update_metrics(metrics, action_taken):
    metrics['total_steps'] += 1
    if action_taken == 'vacuum':
        metrics['vacuums'] += 1
        metrics['energy_consumed'] += 2
    else:
        metrics['moves'] += 1
        metrics['energy_consumed'] += 1
    return metrics

def display_environment(env, agent_pos, action, step, rows, cols):
    header = 'Initial Stage:' if step == 0 else f'Step {step} (Action: {action}):'
    print(header)
    for r in range(rows):
        row_str = ''
        for c in range(cols):
            status = 'D' if env[r][c] == 1 else 'C'
            if (r, c) == agent_pos:
                if action == 'vacuum':
                    cell_repr = f'[A(V),{status}]'
                elif action in ['Up', 'Down', 'Left', 'Right']:
                    abbr = action[0]
                    cell_repr = f'[A({abbr}),{status}]'
                else:
                    cell_repr = f'[A(),{status}]'
            else:
                cell_repr = f'[{status}]'
            row_str += cell_repr
        print(row_str)
    print()

def display_performance_summary(metrics):
    print('=' * 45)
    print('        PERFORMANCE SUMMARY REPORT')
    print('=' * 45)
    print(f"Total Steps Executed:      {metrics['total_steps']}")
    print(f"Total Clean/Vacuum Actions:{metrics['vacuums']}")
    print(f"Total Movement Steps:      {metrics['moves']}")
    print(f"Total Energy Consumed:     {metrics['energy_consumed']} units")
    print(f"Theoretical Min Steps:     {metrics['theoretical_min']}")
    if metrics['theoretical_min'] > 0:
        ratio = metrics['total_steps'] / metrics['theoretical_min']
        print(f"Optimality Ratio:          {ratio:.2f} (Actual / Theoretical)")
    print('=' * 45)

### 4. Running Fully Observable Simulation ($3 \times 3$ Grid)

In [4]:
def run_vacuum_world(rows=3, cols=3, max_iterations=50):
    env = create_environment(rows, cols)
    env = place_dirt(env, dirt_prob=0.35)
    agent_pos = place_agent(rows, cols)

    metrics = {
        'total_steps': 0,
        'vacuums': 0,
        'moves': 0,
        'energy_consumed': 0,
        'theoretical_min': compute_theoretical_minimum(env, agent_pos, rows, cols)
    }

    display_environment(env, agent_pos, 'None', 0, rows, cols)

    step = 0
    while not all_clean(env) and step < max_iterations:
        step += 1
        best_act, util = choose_best_action(agent_pos, env, rows, cols)

        if best_act == 'vacuum':
            vacuum(env, agent_pos)
            update_metrics(metrics, 'vacuum')
            display_environment(env, agent_pos, 'vacuum', step, rows, cols)
        else:
            moves = get_possible_moves(agent_pos, rows, cols)
            agent_pos = moves[best_act]
            update_metrics(metrics, best_act)
            display_environment(env, agent_pos, best_act, step, rows, cols)

    if all_clean(env):
        print('Success: All cells in grid are clean! Task Completed.')
    else:
        print('Stopped: Maximum iterations reached.')

    display_performance_summary(metrics)

random.seed(42)
run_vacuum_world(rows=3, cols=3)

Initial Stage:
[A(),C][D][D]
[D][C][C]
[C][D][C]

Step 1 (Action: Down):
[C][D][D]
[A(D),D][C][C]
[C][D][C]

Step 2 (Action: vacuum):
[C][D][D]
[A(V),C][C][C]
[C][D][C]

Step 3 (Action: Right):
[C][D][D]
[C][A(R),C][C]
[C][D][C]

Step 4 (Action: Up):
[C][A(U),D][D]
[C][C][C]
[C][D][C]

Step 5 (Action: vacuum):
[C][A(V),C][D]
[C][C][C]
[C][D][C]

Step 6 (Action: Down):
[C][C][D]
[C][A(D),C][C]
[C][D][C]

Step 7 (Action: Down):
[C][C][D]
[C][C][C]
[C][A(D),D][C]

Step 8 (Action: vacuum):
[C][C][D]
[C][C][C]
[C][A(V),C][C]

Step 9 (Action: Up):
[C][C][D]
[C][A(U),C][C]
[C][C][C]

Step 10 (Action: Up):
[C][A(U),C][D]
[C][C][C]
[C][C][C]

Step 11 (Action: Right):
[C][C][A(R),D]
[C][C][C]
[C][C][C]

Step 12 (Action: vacuum):
[C][C][A(V),C]
[C][C][C]
[C][C][C]

Success: All cells in grid are clean! Task Completed.
        PERFORMANCE SUMMARY REPORT
Total Steps Executed:      12
Total Clean/Vacuum Actions:4
Total Movement Steps:      8
Total Energy Consumed:     16 units
Theoretical Min Steps:

### 5. Running Partially Observable Utility Agent (Belief State)

In [5]:
def run_partially_observable_utility_agent(rows=3, cols=3, max_iterations=60):
    env = create_environment(rows, cols)
    env = place_dirt(env, dirt_prob=0.35)
    agent_pos = place_agent(rows, cols)
    knowledge = [[None for _ in range(cols)] for _ in range(rows)]

    r, c = agent_pos
    knowledge[r][c] = env[r][c]

    step = 0
    print(f'Initial State - Agent at {agent_pos}:')

    def belief_all_clean(k):
        return all(cell == 0 for row in k for cell in row if cell is not None) and                all(cell is not None for row in k for cell in row)

    while not belief_all_clean(knowledge) and step < max_iterations:
        step += 1
        r, c = agent_pos
        knowledge[r][c] = env[r][c]

        if knowledge[r][c] == 1:
            vacuum(env, agent_pos)
            knowledge[r][c] = 0
            action = 'vacuum'
        else:
            moves = get_possible_moves(agent_pos, rows, cols)
            best_move = None
            best_score = float('-inf')
            for act, npos in moves.items():
                nr, nc = npos
                score = 0
                if knowledge[nr][nc] is None:
                    score += 5
                elif knowledge[nr][nc] == 1:
                    score += 10
                else:
                    score -= 1
                if score > best_score:
                    best_score = score
                    best_move = act
            if best_move is None:
                best_move = random.choice(list(moves.keys()))
            agent_pos = moves[best_move]
            action = best_move

        print(f'Step {step}: Action = {action}, Agent Pos = {agent_pos}')

    print('Completed exploration & cleaning in Partially Observable World.')

random.seed(42)
run_partially_observable_utility_agent(rows=3, cols=3)

Initial State - Agent at (0, 0):
Step 1: Action = Down, Agent Pos = (1, 0)
Step 2: Action = vacuum, Agent Pos = (1, 0)
Step 3: Action = Down, Agent Pos = (2, 0)
Step 4: Action = Right, Agent Pos = (2, 1)
Step 5: Action = vacuum, Agent Pos = (2, 1)
Step 6: Action = Up, Agent Pos = (1, 1)
Step 7: Action = Up, Agent Pos = (0, 1)
Step 8: Action = vacuum, Agent Pos = (0, 1)
Step 9: Action = Right, Agent Pos = (0, 2)
Step 10: Action = vacuum, Agent Pos = (0, 2)
Step 11: Action = Down, Agent Pos = (1, 2)
Step 12: Action = Down, Agent Pos = (2, 2)
Step 13: Action = Up, Agent Pos = (1, 2)
Completed exploration & cleaning in Partially Observable World.
